In [1]:
import pandas as pd
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

# 1. Select GPU or CPU for model inference
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Load book metadata and assign unique IDs
books = pd.read_csv('ATML2025_books.csv')
books['BookID'] = books.index

# 3. Extract any metadata prices into a separate table
if 'price' in books.columns:
    books['price'] = pd.to_numeric(books['price'], errors='coerce')       # convert to float, NaNs if invalid
    meta_prices = books[['BookID','price']].dropna().copy()               # only keep real prices
    meta_prices['source'] = 'metadata'
else:
    meta_prices = pd.DataFrame(columns=['BookID','price','source'])

# 4. Load reviews and pull out review‐level prices
reviews_raw = pd.read_csv('ATML2025_book_reviews_train.csv')
rev_price = reviews_raw[['Title','Price']].copy()
rev_price['Price'] = pd.to_numeric(rev_price['Price'], errors='coerce')
rev_price = rev_price.merge(books[['Title','BookID']], on='Title', how='left')
# 5. Compute median review price per book
review_prices = (
    rev_price.dropna(subset=['Price'])
             .groupby('BookID', as_index=False)['Price']
             .median()
             .rename(columns={'Price':'price'})
)
review_prices['source'] = 'reviews'

# 6. Combine metadata + review prices and remove price from main books
prices = pd.concat([meta_prices, review_prices], ignore_index=True)
if 'price' in books.columns:
    books = books.drop(columns=['price'])

# 7. Build normalized tables for SQLite
books_table   = books[['BookID','Title','description','authors','publisher','publishedDate','categories']]
table_prices  = prices[['BookID','price','source']]
reviews_raw['ReviewID'] = reviews_raw.index
reviews = reviews_raw.drop(columns=['Price'], errors='ignore')
reviews_table = reviews.merge(books_table[['BookID','Title']], on='Title', how='left')[
    ['ReviewID','BookID','rating','time','summary','text']
]

# 8. Write tables to SQLite
conn = sqlite3.connect('books.db')
books_table.to_sql('Books', conn, index=False, if_exists='replace')
table_prices.to_sql('Prices', conn, index=False, if_exists='replace')
reviews_table.to_sql('Reviews', conn, index=False, if_exists='replace')
conn.close()

# 9. Generate and persist sentence‐embeddings for all non‐null descriptions
books_with_desc = books[books['description'].notnull()].reset_index(drop=True)
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embeddings = model.encode(books_with_desc['description'].tolist(),
                          show_progress_bar=True,
                          convert_to_numpy=True)
np.save('embeddings.npy', embeddings)
np.save('book_ids.npy', books_with_desc['BookID'].values)


c:\Users\pooya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Using device: cuda


C:\Users\pooya\AppData\Local\Temp\ipykernel_31192\186838454.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prices = pd.concat([meta_prices, review_prices], ignore_index=True)
c:\Users\pooya\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/2457 [00:00<?, ?it/s]

a display of the database to ensure of correct implementation

In [2]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('books.db')

# Retrieve the first 5 rows of the table using pandas
df = pd.read_sql_query('SELECT * FROM books LIMIT 5', conn)

# Display the result
print(df)

# Close the connection
conn.close()

   BookID                                              Title  \
0       0                           Dr. Seuss: American Icon   
1       1              Wonderful Worship in Smaller Churches   
2       2                      Whispers of the Wicked Saints   
3       3  The Church of Christ: A Biblical Ecclesiology ...   
4       4                         The Overbury affair (Avon)   

                                         description  \
0  Philip Nel takes a fascinating look into the k...   
1  This resource includes twelve principles in un...   
2  Julia Thomas finds her life spinning out of co...   
3  In The Church of Christ: A Biblical Ecclesiolo...   
4                                               None   

                    authors                   publisher publishedDate  \
0            ['Philip Nel']                   A&C Black    2005-01-01   
1          ['David R. Ray']                        None          2000   
2       ['Veronica Haddon']                   iUniverse    

In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sqlite3
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime
from nltk.sentiment.vader import SentimentIntensityAnalyzer

class Recommender:
    def __init__(self,
                 db_path='books.db',
                 embed_path='embeddings.npy',
                 ids_path='book_ids.npy',
                 sim_weight=0.7,
                 rating_weight=0.3):
        """
        Initialize the recommender system by loading the book data,
        review data, prices, and sentence embeddings.
        Also prepares sentiment analyzer and category cleanup.
        """

        # Load data from SQLite database
        conn = sqlite3.connect(db_path)
        self.books = pd.read_sql('SELECT * FROM Books', conn)
        self.prices = pd.read_sql('SELECT * FROM Prices', conn)
        self.reviews = pd.read_sql('SELECT * FROM Reviews', conn)
        conn.close()

        # Precompute average ratings per book
        self.avg_rating = (
            self.reviews.groupby('BookID')['rating']
                        .mean()
                        .rename('avg_rating')
                        .reset_index()
        )

        # Load precomputed sentence embeddings and corresponding book IDs
        self.embeddings = np.load(embed_path)
        self.book_ids   = np.load(ids_path)

        # Load the sentence transformer model for encoding text queries
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

        # Sentiment analyzer for detecting sentiment of the query
        self.sent_analyzer = SentimentIntensityAnalyzer()

        # Weight settings for combining similarity and rating scores
        self.sim_w = sim_weight
        self.rating_w = rating_weight

        # Prepare book categories and publication dates
        self.books['categories_clean'] = (
            self.books['categories']
            .fillna('')
            .str.replace(r"[\[\]']", "", regex=True)  # Clean brackets and quotes
            .str.lower()
        )
        self.books['pub_dt'] = pd.to_datetime(
            self.books['publishedDate'], errors='coerce'
        )

    def parse_query(self, query):
        """
        Parse a user query to extract filters such as genre, max price,
        publisher, date range, keywords, and sentiment.
        """
        q_low = query.lower()
        sent = self.sent_analyzer.polarity_scores(query)['compound']

        # Initialize filters
        genre = None; max_price = None; publisher = None
        date_before = None; date_after = None

        # Extract genre from query
        m = re.search(r"\b(mystery|fantasy|fiction|romance|thriller|science fiction|history|biography|non-fiction)\b", q_low)
        if m: genre = m.group(0)

        # Extract maximum price
        m = re.search(r"under\s*\$(\d+)", q_low)
        if m: max_price = float(m.group(1))

        # Extract publisher
        m = re.search(r"by\s+([A-Za-z0-9 &]+)", query)
        if m: publisher = m.group(1).strip()

        # Extract year constraints
        m = re.search(r"before\s+(\d{4})", q_low)
        if m: date_before = datetime(int(m.group(1)),1,1)
        m = re.search(r"after\s+(\d{4})", q_low)
        if m: date_after  = datetime(int(m.group(1)),1,1)

        # Extract remaining keywords (excluding stopwords)
        tokens = re.findall(r"\w+", query)
        stop = ['under','by','books','book','before','after'] + ([genre] if genre else [])
        keywords = [w for w in tokens if w.lower() not in stop]

        return genre, max_price, publisher, date_before, date_after, keywords, sent

    def filter_books(self, genre, max_price, publisher, date_before, date_after):
        """
        Filter books based on extracted metadata from user query.
        """
        df = self.books.copy()
        if genre:
            df = df[df['categories_clean'].str.contains(genre, na=False)]
        if publisher:
            df = df[df['publisher'].str.contains(publisher, case=False, na=False)]
        if date_before is not None:
            df = df[df['pub_dt'] <= date_before]
        if date_after is not None:
            df = df[df['pub_dt'] >= date_after]
        if max_price is not None:
            # Filter by books that have a price under the threshold
            valid_ids = self.prices[self.prices['price'] < max_price]['BookID'].unique()
            df = df[df['BookID'].isin(valid_ids)]
        return df

    def get_recommendations(self, query, top_k=5):
        """
        Main function to get top K book recommendations based on the input query.
        Combines semantic similarity and average user rating.
        """
        # Step 1: Parse query
        genre, max_price, publisher, dbefore, dafter, keywords, user_sent = self.parse_query(query)

        # Step 2: Filter dataset
        candidates = self.filter_books(genre, max_price, publisher, dbefore, dafter)
        if candidates.empty:
            return f"No books found for '{query}'"

        # Step 3: Compute similarity between query and filtered books
        q_emb = self.model.encode(query, convert_to_numpy=True)

        # Step 4: Subset the embeddings to candidate books
        mask = np.isin(self.book_ids, candidates.BookID.values)
        sub_embs = self.embeddings[mask]
        sub_ids  = self.book_ids[mask]

        # Step 5: Compute cosine similarity
        sims = cosine_similarity(q_emb.reshape(1,-1), sub_embs).flatten()

        # Step 6: Retrieve average ratings
        rating_map = self.avg_rating.set_index('BookID')['avg_rating'].to_dict()

        # Step 7: Score books by combining similarity and rating
        scored = []
        for bid, sim in zip(sub_ids, sims):
            rating = rating_map.get(bid, np.nan)
            r_norm = rating/5 if not np.isnan(rating) else 0
            final_score = self.sim_w * sim + self.rating_w * r_norm
            scored.append((bid, final_score, sim, rating))

        # Step 8: Select top K books
        top = sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]
        rec_ids = [b for b,_,_,_ in top]
        sim_map = {b: s for b,_,s,_ in top}

        # Step 9: Merge with metadata for final output
        recs = self.books[self.books.BookID.isin(rec_ids)].copy()
        recs = recs.set_index('BookID').loc[rec_ids].reset_index()
        recs['similarity'] = recs.BookID.map(sim_map)
        recs = recs.merge(self.avg_rating, on='BookID', how='left')
        recs = recs.merge(self.prices[['BookID','price']], on='BookID', how='left')

        # Step 10: Format output as list of dictionaries
        output = []
        for _, row in recs.iterrows():
            output.append({
                'Title': row.Title,
                'authors': row.authors,
                'price': row.price,
                'avg_rating': row.avg_rating,
                'similarity': row.similarity,
                'publishedDate': row.publishedDate
            })
        return output

# -------------------------------
# UI Setup using IPython widgets
# -------------------------------

# Instantiate the recommender
recommender = Recommender()

# Define widgets
query_input = widgets.Text(placeholder='e.g., a mystery book')
submit_btn = widgets.Button(description='Submit', icon='check')
clear_btn  = widgets.Button(description='Clear', icon='eraser')
output     = widgets.Output()

def on_submit(b):
    """
    Callback function triggered when user submits a query.
    Displays recommendations or error message.
    """
    with output:
        output.clear_output()
        q = query_input.value.strip()
        if not q:
            print("Enter a query.")
            return
        recs = recommender.get_recommendations(q)
        if isinstance(recs, str):
            print(recs)
        else:
            for i, r in enumerate(recs, start=1):
                price = f"${r['price']:.2f}" if pd.notna(r['price']) else "N/A"
                rating = f"{r['avg_rating']:.1f}" if pd.notna(r['avg_rating']) else "N/A"
                print(f"{i}. {r['Title']} — {r['authors']} | {price} | Rating: {rating} | Sim: {r['similarity']:.2f}")

# Bind buttons to actions
submit_btn.on_click(on_submit)
clear_btn.on_click(lambda b: output.clear_output())

# Display the widgets
display(query_input, submit_btn, clear_btn, output)


c:\Users\pooya\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Text(value='', placeholder='e.g., a mystery book')

Button(description='Submit', icon='check', style=ButtonStyle())

Button(description='Clear', icon='eraser', style=ButtonStyle())

Output()